In [0]:
%sql
DESCRIBE samples.nyctaxi.trips;

#Practica de Ctes y window functions

In [0]:
%sql
With paso1 AS(
    SELECT AVG(trip_distance) as distancia_por_zona_de_inicio, AVG(fare_amount) as Promedio_tarifa, Count(*) as cantidad_viajes
    FROM samples.nyctaxi.trips
    Group BY pickup_zip
    HAVING cantidad_viajes>100    
)

SELECT * 
FROM paso1



Compara el promedio de
tarifa por hora del día con el
promedio general. Usa dos CTEs: una
para calcular el promedio por hora,
otra para calcular el promedio
general, y luego únelas para mostrar
la diferencia

In [0]:
%sql
WITH promedio_por_hora AS (
    select AVG(fare_amount) as promedio_tarifa
    from samples.nyctaxi.trips
    GROUP BY HOUR(tpep_pickup_datetime)
), promedio AS(
    SELECT AVG(fare_amount) as promedio
    FROM samples.nyctaxi.trips
)
SELECT (promedio_por_hora.promedio_tarifa-promedio.promedio) as diferencia, promedio, promedio_tarifa as por_hora
FROM promedio_por_hora, promedio


Calcula estadísticas de
distancia solo para viajes válidos
(distancia > 0, tarifa > 0). Usa una
CTE para filtrar primero los datos
válidos, y luego calcula mínimo,
máximo, promedio y mediana.

In [0]:
%sql
With datos_validos As (
    Select *
    FROM samples.nyctaxi.trips
    Where trip_distance>0 AND fare_amount>0
)

Select MIN(trip_distance) as minimo, MAX(trip_distance) as maximo, AVG(trip_distance) as promedio, MEDIAN(trip_distance) as mediana
From datos_validos

 Crea un ranking de los viajes más caros.
Muestra los top 20 viajes con su fecha, distancia,
tarifa y el ranking (1 = más caro).

In [0]:
%sql
with rn as (
Select trip_distance as distancia, fare_amount as tarifa, tpep_pickup_datetime as fecha, ROW_NUMBER() OVER (ORDER BY fare_amount DESC) as rown
From samples.nyctaxi.trips)

Select * 
From rn
where rown<=20



Para las 10 zonas con más viajes, crea tres
rankings diferentes usando ROW_NUMBER(), RANK() y
DENSE_RANK() ordenados por cantidad de viajes. ¿Notas la
diferencia?


In [0]:
%sql
With zonas as(
    select Count(*) as cantidad_viajes, pickup_zip
    from samples.nyctaxi.trips
    Group by pickup_zip
    HAVING cantidad_viajes>10
), rn as (
    select ROW_NUMBER() OVER(ORDER BY cantidad_viajes DESC) as rown, pickup_zip, cantidad_viajes
    from zonas
), rank as (
    select RANK() OVER(ORDER BY cantidad_viajes DESC) as rank, pickup_zip, cantidad_viajes
    from zonas
), dr as (
    SELECT DENSE_RANK() OVER(ORDER BY cantidad_viajes DESC) as dr, pickup_zip, cantidad_viajes
    from zonas
)

SELECT 
    rn.rown,
    rank.rank,
    dr.dr,
    rn.pickup_zip,
    rn.cantidad_viajes
FROM rn
JOIN rank ON rn.pickup_zip = rank.pickup_zip
JOIN dr ON rn.pickup_zip = dr.pickup_zip
ORDER BY rn.rown



    


#Window Functions Avanzadas

In [0]:
%sql
SELECT 
    fare_amount AS tarifa,
    trip_distance AS distancia,
    fare_amount - AVG(fare_amount) OVER() AS diferencia,
    100 * (fare_amount - AVG(fare_amount) OVER()) / AVG(fare_amount) OVER() AS porcentaje_diferencia
FROM samples.nyctaxi.trips

In [0]:
%sql
SELECT tpep_pickup_datetime, fare_amount, pickup_zip, AVG(fare_amount) OVER(PARTITION BY pickup_zip) AS tarifa_zona
FROM samples.nyctaxi.trips

In [0]:
%sql
Select tpep_pickup_datetime, fare_amount, LAG(fare_amount) OVER(ORDER BY tpep_pickup_datetime) as anterior
From samples.nyctaxi.trips

In [0]:
%sql
SELECT SUM(fare_amount) OVER(ORDER BY tpep_pickup_datetime ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS acumulado, tpep_pickup_datetime, fare_amount
FROM samples.nyctaxi.trips

In [0]:
%sql
WITH valido as(
    SELECT *
    FROM samples.nyctaxi.trips
    WHERE fare_amount>0 AND trip_distance>0
), estadisticas_zona AS (
    Select AVG(fare_amount) as Promedio, MAX(fare_amount) as maximo, MIN(fare_amount) as minimo, pickup_zip
    FROM valido
    Group BY pickup_zip
), con_rango AS (
    SELECT DENSE_RANK() OVER(ORDER BY Promedio DESC) as Rango, Promedio, pickup_zip
    FROM estadisticas_zona
)
SELECT *
FROM con_rango
WHERE Rango<=10
ORDER BY Rango ASC